# Data Cleaning

In this notebook, we will begin cleaning up our IMDb and Oscar datasets in order to create visualizations that will help invesitgating our narrative.

## Preprocessing

We only need to import Pandas for reading in .csv files, creating dataframes, and to mainly help with cleaning the data. The AST library is for abstract syntax trees, which can help with changing datatypes from strings to lists when using Regex is not possible (this problem will be shown later below).

In [1]:
import pandas as pd
import ast

### Initializing helper functions
We created a file with functions that help read and write data after filtering, which we initialize at the top of our program with Jupyter Magics.

In [2]:
%run helpers.ipynb

### Reading in CSV files

Here we will read in the CSV files for the IMDb and Oscar datasets and print out a preview of how their raw dataframes look like.

In [3]:
raw_imdb_df = pd.read_csv("data/imdb_full_data.csv")
raw_imdb_df.head()

,id,title,duration,mpa,rating,votes,méta_score,description,movie_link,writers,...,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,filming_locations,production_companies,awards_content,genres,languages
0,tt0027483,The Crimson Circle,1h 16m,NaN,6.4,30,NaN,An extortion ring murders anyone who refuses t...,https://www.imdb.com/title/tt0027483/,"['Reginald Denham', 'Edgar Wallace', 'Howard I...",...,NaN,NaN,NaN,1936-08-10,['United Kingdom'],NaN,['Richard Wainwright Productions'],NaN,['Drama'],['English']
1,tt0058131,The Mystery of Thug Island,1h 36m,NaN,5.0,114,NaN,"Three year old Ada, daughter of the British ca...",https://www.imdb.com/title/tt0058131/,"['Emilio Salgari', 'Arpad DeRiso', 'Ottavio Po...",...,NaN,NaN,NaN,1966-05-28,"['Italy', 'Monaco', 'West Germany']",NaN,"['Eichberg-Film', 'Liber Film']",NaN,['Adventure'],['Italian']
2,tt0042760,Las mujeres de mi general,1h 52m,Not Rated,6.8,74,NaN,Infante stars as a rebel general caught up in ...,https://www.imdb.com/title/tt0042760/,"['Joselito Rodríguez', 'Celestino Gorostiza', ...",...,NaN,NaN,NaN,1951-07-13,['Mexico'],NaN,['Producciones Rodríguez Hermanos'],NaN,"['Drama', 'War']",['Spanish']
3,tt0027667,Gentle Julia,1h 2m,Approved,6.8,38,NaN,A shy newspaperman (Brown) nearly gives up whe...,https://www.imdb.com/title/tt0027667/,"['Booth Tarkington', 'Lamar Trotti']",...,NaN,NaN,NaN,1936-04-10,['United States'],"['20th Century Fox Studios - 10201 Pico Blvd.,...",['Twentieth Century Fox'],NaN,"['Comedy', 'Drama', 'Romance']",['English']
4,tt0055747,Love at Twenty,1h 50m,NaN,7.2,2.5K,NaN,"""Love at Twenty"" unites five directors from ar...",https://www.imdb.com/title/tt0055747/,"['Shintarô Ishihara', 'Marcel Ophüls', 'Renzo ...",...,NaN,NaN,NaN,1963-02-06,"['France', 'Italy', 'Japan', 'Poland', 'West G...","['Warsaw Zoo, Ratuszowa, Praga Pólnoc, Warsaw,...","['Ulysse Productions', 'Unitec Films', 'Cinese...",NaN,"['Drama', 'Romance']","['French', 'Polish', 'Japanese', 'Italian', 'G..."


In [4]:
raw_oscar_df = pd.read_csv("data/oscar_full_data.csv", sep="\t")
raw_oscar_df.head()

,Ceremony,Year,Class,CanonicalCategory,Category,NomId,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation,MultifilmNomination
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051251,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins,NaN,NaN,True
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051252,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,The Patent Leather Kid,NaN,NaN,True
2,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250a,The Last Command,tt0019071,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexander],NaN,NaN,True
3,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,an0051250b,The Way of All Flesh,tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,August Schilling,NaN,NaN,True
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,an0051255,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN,NaN


In [5]:
print("Raw IMDb dataset shape: ", raw_imdb_df.shape)
print("Raw Oscar dataset shape: ", raw_oscar_df.shape)

Raw IMDb dataset shape:  (63249, 23)
Raw Oscar dataset shape:  (12014, 16)


## Filtering Columns - IMDb data

We want to begin dropping any columns with irrelevant information that our IMDb dataset provides us. These include `MPA` (Motion Picture Association), `film_duration`, `description`, `movie_link` (IMDb URL), and `filming_locations`. These columns are fine to drop since they won't be needed for our investigation since they are only basic and additional film details. In addition, the `awards_content` column was dropped since all the row values are empty.

The next step is to drop any empty row values in the IMDb rating or user votes columns. These two fields will be helpful for measuring audience interest.

Finally, we can go ahead and sort all of the films in the IMDb dataset by release date, starting with the earliest (from 1920 to 2025).

In [6]:
# keep all relevant rows in imdb dataset
imdb_df = raw_imdb_df.drop(columns=['mpa', 'duration', 'description', 'movie_link', 'filming_locations', 'awards_content'])

# filter out NaNs in rating or votes columns
imdb_df = imdb_df.dropna(subset=['rating', 'votes'], how='any')

# sort release date by ascending order (convert column from string to datetime)
imdb_df = imdb_df.assign(release_date=pd.to_datetime(imdb_df['release_date'])).sort_values(by='release_date').reset_index(drop=True)

imdb_df.head()

,id,title,rating,votes,méta_score,writers,directors,stars,budget,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,production_companies,genres,languages
0,tt0011370,Klostret i Sendomir,6.9,485,NaN,"['Franz Grillparzer', 'Victor Sjöström']",['Victor Sjöström'],"['Tore Svennberg', 'Tora Teje', 'Richard Lund'...",NaN,NaN,NaN,NaN,1920-01-01,['Sweden'],['Svenska Biografteatern AB'],['Drama'],['None']
1,tt0011413,The Lost City,4.7,35,NaN,['Frederick Chapin'],['E.A. Martin'],"['Juanita Hansen', 'George Chesebro', 'Frank C...",NaN,NaN,NaN,NaN,1920-01-01,['United States'],['Selig Polyscope Company'],"['Action', 'Adventure']","['None', 'English']"
2,tt0276209,Hypnose,7.0,19,NaN,['Karl Schneider'],['Richard Eichberg'],"['Lee Parry', 'Gertrud de Lalsky', 'Karl Halde...",NaN,NaN,NaN,NaN,1920-01-03,['Germany'],['Richard Eichberg-Film GmbH'],"['Drama', 'Mystery']",['None']
3,tt0010495,My Husband's Other Wife,5.3,17,NaN,['Stanley Olmstead'],['J. Stuart Blackton'],"['Sylvia Breamer', 'Robert Gordon', 'Warren Ch...",NaN,NaN,NaN,NaN,1920-01-04,['United States'],['J. Stuart Blackton Feature Pictures'],"['Drama', 'Romance']",['None']
4,tt0010502,Nachtgestalten,5.6,25,NaN,"['Richard Oswald', 'Karl Hans Strobl']",['Richard Oswald'],"['Paul Wegener', 'Reinhold Schünzel', 'Erna Mo...",NaN,NaN,NaN,NaN,1920-01-09,['Germany'],['Richard-Oswald-Produktion'],['Horror'],"['None', 'German']"


We've managed to filter out around 4,000 rows and 6 columns in the IMDb dataset so far, as seen by the shape of the dataframe.

In [7]:
print("Cleaned IMDb dataset shape: ", imdb_df.shape)

Cleaned IMDb dataset shape:  (59181, 17)


## Filtering Columns - Oscar data

For the Oscar nominee dataset, we will also filter out any irrelevant columns. The fields `NomId` and `NomineeIds` can be dropped because these IDs do not appear in the IMDb dataset and would not help with comparison across both datasets. Instead, we want to join the two datasets together later on using `FilmId` (called `id` in the IMDb dataset), so we must filter out any empty row values in that column.

`Detail` gives information about a character's name in a film, `Note` is additional information prodived about the award, and `Citation` is the official text of the award statement. These columns don't have any use in our investigation and mostly contain empty row values, so they can be dropped. Only 40 entries in the Oscar dataset had data for `MultifilmNomination`, so we chose to drop it.

The field `Year` will be converted from strings to floats. One thing to note is that the first 6 years of the Oscar Awards spanned 12+ months, which is why the periods from 1927-1933 were displayed as 'YYYY/YY' like '1927/28'. The Oscar's did not become a calendar year event until 1934, therefore the logic in the code below for converting these years will reflect that.

Finally, we want to focus this investigation exclusively on feature films, so short film formats or documentaries will not be joined. Therefore, we decided to drop any rows in the dataset that contained the words 'Documentary' or 'Short' in their respective award categories.

The Oscars also contain nomination categories that are not specific to a certain film (honorary or special awards), so simply dropping rows where `FilmId` is empty filters them out of the dataset.

In [8]:
# keep all relevant rows in oscar dataset
oscar_df = raw_oscar_df.drop(columns=['NomId', 'NomineeIds', 'Detail', 'Note', 'Citation', 'MultifilmNomination'])

# filter out NaNs in FilmId
oscar_df = oscar_df.dropna(subset=['FilmId'])

# filter out irrelevant nomination categories in oscar dataset 
# documentary, short-form, special-award, or legacy categories
oscar_df = oscar_df[~oscar_df['Category'].str.lower().str.contains("documentary|short", na=False)]

# convert years from strings to ints
# Note: the Oscar's did not become a calendar year event until 1934
x = oscar_df['Year'].astype(str)

oscar_df['Year'] = (
    oscar_df['Year']
    .astype(str)
    .str[:4] # take first 4 digits of year
    .astype(int)
    .add(x.str.contains('/').astype(int)) # add one to year if '/' is present (exaA: 1927/28 will become 1928)
)

oscar_df.head()

,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,Winner
0,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose,tt0019217,Richard Barthelmess,Richard Barthelmess,NaN
1,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Patent Leather Kid,tt0018253,Richard Barthelmess,Richard Barthelmess,NaN
2,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command,tt0019071,Emil Jannings,Emil Jannings,True
3,1,1928,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Way of All Flesh,tt0019553,Emil Jannings,Emil Jannings,True
4,1,1928,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,NaN


We've managed to filter out around 3,000 rows and 6 columns in the Oscar dataset so far, as seen by the shape of the dataframe.

In [9]:
print("Cleaned Oscar dataset shape: ", oscar_df.shape)

Cleaned Oscar dataset shape:  (9073, 10)


## Converting datatypes - IMDb data

Nearly all of the data in the IMDb dataset are strings, so we went ahead and assigned them to their appropriate datatypes.

The column `meta_score` was renamed to be indexed without the grave accent, and the row values were converted into integers. The `Int64` datatype was used because it's a nullable integer and can handle the empty row values in the dataset. The row values in the `rating` column were converted to floats.

For the `votes` column, many user votes in the triple and quadruple digits showed up as strings like "9K" for 9,000 votes and "1.1M" for 1,100,000 votes. Therefore, we used Regex to replace these characters with numerical values and make sure that the row values were converted to integers. The `Int64` datatype is used again to handle empty row values.

Lastly, we converted any columns with row values that contained multiple elements, such as `genre` containing `[Drama, Mystery]`. This appears like a list, but is actually a string, so we leveraged the AST (Abstract Syntax Tree) library to convert these strings to lists. We also applied this method to the `writers`, `directors`, `stars`, `countries_origin`, `production_companies`, and `languages` columns. One limitation with this task was that some actors, writers, and directors could have names with apostrophes in them, which the AST library also converts properly.

In [10]:
# convert meta_score/ratings in imdb dataset from strings to ints/floats
imdb_df = imdb_df.rename(columns={'méta_score': 'meta_score'})
imdb_df['meta_score'] = pd.to_numeric(imdb_df['meta_score']).astype('Int64') # nullable integer type (use since some meta_scores are NaN)
imdb_df['rating'] = pd.to_numeric(imdb_df['rating'])

# convert votes in imdb dataset from strings (ex: 9K, 1.1M) to ints
imdb_df['votes'] = (
    imdb_df['votes']
    .astype(str)
    .str.replace(r'K$', 'e3', regex=True)
    .str.replace(r'M$', 'e6', regex=True)
    .apply(pd.to_numeric) # convert to float to translate e3/e6 into digits
    .astype(int) # use instead of Int64 since NaNs have been dropped already
)

# clean up text formatting for columns in imdb database (convert from strings to lists)
imdb_df[['writers', 'directors', 'stars', 'countries_origin', 'production_companies', 'genres', 'languages']] = (
    imdb_df[['writers', 'directors', 'stars', 'countries_origin', 'production_companies', 'genres', 'languages']]
    .apply(
        lambda col: col.map(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    )
)

# print out data types of each variable in imdb dataframe
imdb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 59181 entries, 0 to 59180
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     59181 non-null  str           
 1   title                  59181 non-null  str           
 2   rating                 59181 non-null  float64       
 3   votes                  59181 non-null  int64         
 4   meta_score             15531 non-null  Int64         
 5   writers                59000 non-null  object        
 6   directors              59168 non-null  object        
 7   stars                  58920 non-null  object        
 8   budget                 15281 non-null  str           
 9   opening_weekend_gross  16799 non-null  str           
 10  gross_worldwide        20670 non-null  str           
 11  gross_us_canada        19459 non-null  str           
 12  release_date           59181 non-null  datetime64[us]
 13  countries_or

### Writing to CSV

To reuse our filtered data in other notebooks for joining and visualizing, we use our helper function to write the data frames to CSV files that can be read in later.

In [11]:
write_dfs(oscar_df, imdb_df)